# AMP Finder AI — Biological Understanding

## Goal

Inspect what separates the operational AMP and putative-non-AMP labels before training any model. The notebook focuses on charge, hydropathy, length, amphipathicity-related descriptors, and amino-acid composition.

If the real processed dataset is absent, the notebook uses the bundled synthetic UI dataset and labels every result as a technical demonstration.

## Setup

For Colab, clone the repository and install `requirements-dev.txt` first. Raw APD data are not bundled; follow `docs/DATA_ACQUISITION.md`.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src/amp_finder").exists():
            return candidate
    raise FileNotFoundError("Run this notebook inside the AMP Finder AI project.")

PROJECT_ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
(PROJECT_ROOT / "outputs/figures").mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from amp_finder.constants import AMINO_ACIDS
from amp_finder.features import extract_feature_frame

sns.set_theme(style="whitegrid", context="notebook")
real_dataset_path = PROJECT_ROOT / "data/processed/amp_dataset.csv"
demo_dataset_path = PROJECT_ROOT / "data/demo/demo_sequences.csv"
dataset_path = real_dataset_path if real_dataset_path.exists() else demo_dataset_path
dataset = pd.read_csv(dataset_path)
is_demo = dataset_path == demo_dataset_path
print("Dataset:", dataset_path)
print("Status:", "SYNTHETIC UI DEMO — NOT SCIENTIFIC EVIDENCE" if is_demo else "Prepared APD/UniProt demonstration data")
print("Rows:", len(dataset))
dataset.head()

## Steps

### 1. Check label, sequence, and split integrity

In [ ]:
required = {"sequence", "label", "split", "split_group"}
assert required.issubset(dataset.columns), f"Missing: {required - set(dataset.columns)}"
assert set(dataset["label"].unique()) == {0, 1}
assert not dataset["sequence"].duplicated().any(), "Exact sequence duplicates found"
assert (dataset.groupby("split_group")["split"].nunique() == 1).all(), "Group leakage found"

quality_table = (
    dataset.groupby(["split", "label"], observed=True)
    .size()
    .rename("rows")
    .reset_index()
)
quality_table["class"] = quality_table["label"].map({1: "AMP", 0: "Putative non-AMP"})
quality_table

### 2. Extract explainable biological features

In [ ]:
features = extract_feature_frame(dataset["sequence"])
overlapping_feature_columns = [column for column in features.columns if column in dataset.columns]
dataset_context = dataset.drop(columns=overlapping_feature_columns).reset_index(drop=True)
analysis = pd.concat([dataset_context, features], axis=1)
analysis["class"] = analysis["label"].map({1: "AMP", 0: "Putative non-AMP"})

summary_columns = ["length", "net_charge_pH7", "gravy", "hydrophobic_moment", "aromaticity", "isoelectric_point"]
summary = analysis.groupby("class", observed=True)[summary_columns].agg(["median", "mean", "std"])
summary.round(3)

### 3. Compare the main feature distributions

In [ ]:
palette = {"AMP": "#2F6BFF", "Putative non-AMP": "#D59B2D"}
figure, axes = plt.subplots(2, 2, figsize=(12, 8))
for axis, feature, title in zip(
    axes.ravel(),
    ["length", "net_charge_pH7", "gravy", "hydrophobic_moment"],
    ["Sequence length", "Estimated net charge at pH 7", "Mean hydropathy (GRAVY)", "Alpha-helical hydrophobic moment"],
):
    sns.histplot(
        data=analysis,
        x=feature,
        hue="class",
        hue_order=["AMP", "Putative non-AMP"],
        palette=palette,
        element="step",
        stat="density",
        common_norm=False,
        ax=axis,
        legend=(feature == "net_charge_pH7"),
    )
    axis.set_title(title)
    axis.set_xlabel(title)
    axis.set_ylabel("Density")
    if feature == "net_charge_pH7" and axis.get_legend() is not None:
        axis.get_legend().set_title("Operational label")
figure.suptitle("Biological feature distributions by operational label", fontsize=15, y=1.02)
figure.tight_layout()
overview_path = PROJECT_ROOT / "outputs/figures/01_biological_overview.png"
figure.savefig(overview_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved:", overview_path)

### 4. Examine charge and hydropathy together

In [ ]:
figure, axis = plt.subplots(figsize=(8.5, 6))
sns.scatterplot(
    data=analysis,
    x="gravy",
    y="net_charge_pH7",
    hue="class",
    hue_order=["AMP", "Putative non-AMP"],
    palette=palette,
    alpha=0.70,
    s=48,
    ax=axis,
)
axis.axhline(0, color="#667085", linewidth=1, linestyle="--")
axis.set_title("Charge–hydropathy sequence space")
axis.set_xlabel("Mean hydropathy (Kyte–Doolittle GRAVY)")
axis.set_ylabel("Estimated net charge at pH 7")
axis.legend(title="Operational label", frameon=False)
figure.tight_layout()
scatter_path = PROJECT_ROOT / "outputs/figures/01_charge_hydropathy.png"
figure.savefig(scatter_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved:", scatter_path)

### 5. Compare amino-acid composition

In [ ]:
composition_columns = [f"aa_{aa}" for aa in AMINO_ACIDS]
composition_means = analysis.groupby("class", observed=True)[composition_columns].mean().T
composition_means.index = [name.replace("aa_", "") for name in composition_means.index]
composition_difference = (
    composition_means["AMP"] - composition_means["Putative non-AMP"]
).sort_values()

figure, axis = plt.subplots(figsize=(9, 6))
colors = ["#D59B2D" if value < 0 else "#2F6BFF" for value in composition_difference]
axis.barh(composition_difference.index, composition_difference.values, color=colors)
axis.axvline(0, color="#344054", linewidth=1)
axis.set_title("Mean amino-acid composition difference")
axis.set_xlabel("AMP minus putative-non-AMP fraction")
axis.set_ylabel("Residue")
figure.tight_layout()
composition_path = PROJECT_ROOT / "outputs/figures/01_composition_difference.png"
figure.savefig(composition_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved:", composition_path)

## Checks

The code below confirms that feature rows are complete and physically plausible. It does not establish that the labels are biologically perfect.

In [ ]:
assert len(features) == len(dataset)
assert np.isfinite(features.to_numpy()).all()
assert (features["length"] == dataset["sequence"].str.len()).all()
composition_sum = features[[f"aa_{aa}" for aa in AMINO_ACIDS]].sum(axis=1)
assert np.allclose(composition_sum, 1.0, atol=1e-8)
print("Feature integrity checks passed.")

if is_demo:
    print("IMPORTANT: observed class differences were deliberately created in synthetic data and are not biological findings.")
else:
    print("Interpret differences as dataset associations, not universal AMP mechanisms.")

## Next Steps

1. If this notebook used synthetic data, prepare the real APD/UniProt dataset and rerun it.
2. Describe distributions and overlap, not only class means.
3. Run `02_baseline_random_forest.ipynb` on the same fixed partitions.
4. Preserve the figures and metadata with the model card.